In [2]:
from __future__ import annotations

from pathlib import Path

import numpy as np


trial_path: Path = Path(
    "../dataset/processed/trials/s21/trial_02.npz"
)

data = np.load(trial_path, allow_pickle=True)

signals = data["signals"]
times = data["times"]
channels = data["channels"]
sfreq = float(data["sfreq"][0])

print("signals shape:", signals.shape)
print("times shape:", times.shape)
print("sfreq:", sfreq)
print("first time:", times[0])
print("last time:", times[-1])
print("expected samples:", int(68 * sfreq))
print("actual samples:", signals.shape[1])
print("channels:", channels[:10])

signals shape: (44, 8704)
times shape: (8704,)
sfreq: 128.0
first time: 0.0
last time: 67.9921875
expected samples: 8704
actual samples: 8704
channels: ['Fp1' 'AF3' 'F3' 'F7' 'FC5' 'FC1' 'C3' 'T7' 'CP5' 'CP1']


In [13]:
import numpy as np
from pathlib import Path

trial_path = Path("../dataset/processed/trials/s21/trial_02.npz")
data = np.load(trial_path, allow_pickle=True)

signals = data["signals"]
channels = data["channels"]
sfreq = float(data["sfreq"][0])

after_start = int(65 * sfreq)
after_end = int(68 * sfreq)

for channel_name in ["Fp1", "Resp", "Plet", "Temp"]:
    idx = list(channels).index(channel_name)
    after_values = signals[idx, after_start:after_end]

    print(channel_name)
    print("shape:", after_values.shape)
    print("all nan:", np.all(np.isnan(after_values)))
    print("nan count:", np.isnan(after_values).sum())
    print("first 5:", after_values[:5])
    print("last 5:", after_values[-5:])
    print()

Fp1
shape: (384,)
all nan: True
nan count: 384
first 5: [nan nan nan nan nan]
last 5: [nan nan nan nan nan]

Resp
shape: (384,)
all nan: True
nan count: 384
first 5: [nan nan nan nan nan]
last 5: [nan nan nan nan nan]

Plet
shape: (384,)
all nan: True
nan count: 384
first 5: [nan nan nan nan nan]
last 5: [nan nan nan nan nan]

Temp
shape: (384,)
all nan: True
nan count: 384
first 5: [nan nan nan nan nan]
last 5: [nan nan nan nan nan]



In [14]:
from __future__ import annotations

import json
from pathlib import Path


events_path: Path = Path("../dataset/processed/events/s21_events.json")

with events_path.open("r", encoding="utf-8") as file:
    events_data = json.load(file)

trial_02 = events_data["trials"][1]

trial_02

{'participant_id': 21,
 'trial': 2,
 'experiment_id': 19,
 'valence': 7.26,
 'arousal': 6.29,
 'dominance': 7.72,
 'liking': 6.78,
 'familiarity': 5,
 'before_start_sample_512': 111655,
 'during_start_sample_512': 114231,
 'after_start_sample_512': 143055,
 'after_end_sample_512': 144591,
 'processed_before_start_sec': 0.0,
 'processed_during_start_sec': 5.0,
 'processed_after_start_sec': 65.0,
 'processed_end_sec': 68.0,
 'expected_samples': 8704,
 'actual_samples': 8704,
 'has_padding_nan': True,
 'trial_npz': 'processed/trials/s21/trial_02.npz',
 'metrics_json': 'processed/metrics/s21/trial_02_metrics.json'}

In [15]:
before = trial_02["before_start_sample_512"]
during = trial_02["during_start_sample_512"]
after = trial_02["after_start_sample_512"]
after_end = trial_02["after_end_sample_512"]

print("before -> during samples:", during - before)
print("during -> after samples:", after - during)
print("after -> after_end samples:", after_end - after)

print("before -> during seconds:", (during - before) / 512)
print("during -> after seconds:", (after - during) / 512)
print("after -> after_end seconds:", (after_end - after) / 512)
print("total seconds:", (after_end - before) / 512)

before -> during samples: 2576
during -> after samples: 28824
after -> after_end samples: 1536
before -> during seconds: 5.03125
during -> after seconds: 56.296875
after -> after_end seconds: 3.0
total seconds: 64.328125


In [20]:
from __future__ import annotations

from pathlib import Path
from typing import Dict

import mne
import numpy as np


BDF_PATH: Path = Path("../dataset/raw/bdf/s28.bdf")


def normalize_event_codes(event_codes: np.ndarray) -> np.ndarray:
    normalized_codes: np.ndarray = event_codes.copy()
    mask: np.ndarray = normalized_codes >= 1638144
    normalized_codes[mask] = normalized_codes[mask] - 1638144
    return normalized_codes


raw = mne.io.read_raw_bdf(
    BDF_PATH,
    preload=False,
    verbose=False,
)

print("Num channels:", len(raw.ch_names))
print("Last channels:", raw.ch_names[-5:])

for channel_name in raw.ch_names[-5:]:
    print("\nCHANNEL:", repr(channel_name))

    try:
        events = mne.find_events(
            raw,
            stim_channel=channel_name,
            shortest_event=1,
            verbose=False,
        )

        if events.size == 0:
            print("No events")
            continue

        events[:, 2] = normalize_event_codes(events[:, 2])

        codes, counts = np.unique(events[:, 2], return_counts=True)
        print(dict(zip(codes.tolist(), counts.tolist())))

        relevant = events[np.isin(events[:, 2], [3, 4, 5])]
        print("Relevant count:", len(relevant))
        print("First 30 relevant codes:")
        print(relevant[:30, 2].astype(int).tolist())

        print("Last 30 relevant codes:")
        print(relevant[-30:, 2].astype(int).tolist())

    except Exception as error:
        print("ERROR:", error)

/tmp/ipykernel_24442/3651355954.py:20: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_bdf(
/tmp/ipykernel_24442/3651355954.py:20: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_bdf(


Num channels: 48
Last channels: ['Erg2', 'Resp', 'Plet', 'Temp', '']

CHANNEL: 'Erg2'
No events

CHANNEL: 'Resp'
No events

CHANNEL: 'Plet'
{1: 73}
Relevant count: 0
First 30 relevant codes:
[]
Last 30 relevant codes:
[]

CHANNEL: 'Temp'
{33: 21, 34: 1}
Relevant count: 0
First 30 relevant codes:
[]
Last 30 relevant codes:
[]

CHANNEL: ''
{1: 157, 2: 4, 3: 39, 4: 37, 5: 39, 7: 1, 4194304: 1, 4194305: 11, 4194307: 2, 4194308: 3, 4194309: 3}
Relevant count: 115
First 30 relevant codes:
[3, 5, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5]
Last 30 relevant codes:
[3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5, 3, 4, 5]
